# Le plus grand nombre à 5 chiffres initial

In [1]:
import random

def get_initial_state():
    return [1,1,1,1,1, random.randint(0,9)]

**Question**: Écrire une fonction qui renvoie `r,s_`, qui correspondent au paiement et à l'état suivant tirés selon la distribution $p(\,\cdot\,|s,a)$.

In [2]:
def transition(s,a):
    s_=s.copy()
    state=s_[:5]
    digit_to_change=s[a-1] # index of digit to change
    input_number=s[5] # digit to input

    if digit_to_change==0: # digit is already full
        return 0,state+[random.randint(0,9)]
    
    s_[a-1]=0
    state=s_[:5]
    return input_number*10**(a-1),state+[random.randint(0,9)]  # Piecewise rewarde because gamma=1

**Question**: Implémenter la politique qui renvoie l'action tirée uniformément parmi les emplacements disponibles.

In [3]:
def pi(s):
    action=random.randint(1,5)
    if sum(s[:5])==0:
        return action # if all digits are taken no matter
    
    while s[action-1]==0: # check for non filled digit 
        action=random.randint(1,5)

    return action

**Question**: Compléter la fonction `run_episode`.

In [4]:
def run_episode(policy):
    s = get_initial_state()
    total_reward = 0
    while sum(s[:5])!=0:
        reward,s_=transition(s,policy(s))
        total_reward+=reward
        s=s_
        
    return total_reward

**Question**: Evaluer la politique $\pi$ en faisant tourner un grand nombre d'épisodes.

In [5]:
def average_reward(n=10000,policy=pi):
    total_reward=0
    for _ in range(n):
        total_reward+=run_episode(policy)
    return total_reward/n

average_reward()
# Around 50 000 which is random


49598.5864

**Question**: Proposer une meilleure politique $\pi'$ et l'évaluer.

On propose la politique qui place les chiffres supérieurs à 7 (resp. inférieurs à 2) dans l'emplacement correspondant à la plus grande puissance de 10 disponible (resp. la plus petite), et les autres chiffres sont placés sur un emplacement disponible tiré uniformément.

In [6]:
def pi_star(s):
    free_pos=[index for index,digit in enumerate(s[:5]) if digit==1]

    digit_to_place=s[5] 
    if digit_to_place>=7:
        a =max(free_pos)
    elif digit_to_place<=2:
        a= min(free_pos)
    else:
        n=len(free_pos)
        a= free_pos[random.randint(0,n-1)]

    return a+1

# Le plus grand nombre à 5 chiffres

On considère à nouveau le problème rencontré dans le TD/TP 1. Le but de l'agent est de former un nombre à 5 chiffres, le plus grand possible. On considère 5 emplacements, initialement vides, correspondant aux chiffre des unités, des dizaines, des centaines, des milliers, et des dizaines de milliers. À chaque étape, un chiffre (entre 0 et 9) est tiré uniformément et présenté à l'agent. Celui-ci doit le placer dans l'un des emplacements disponibles. Le nombre est donc formé au bout de 5 étapes.

On modélise le problème par un MDP dont
- l'espace d'états est $\mathcal{S}=\left\{ 0,1 \right\}^5\times \left\{ 0,\dots,9 \right\}$ où pour les cinq premières composantes, un 1 correspond à un emplacement libre,
- l'ensemble de paiements est $\mathcal{R}=\left\{ 0,1,\dots,99999 \right\}$,
- l'ensemble d'actions est $\mathcal{A}=\left\{ 1,\dots,5 \right\}$,
- la dynamique de transition est 
$$p(\,\cdot\,|s,a)= \begin{cases} \delta_0\otimes \delta_s&\text{si $s^{(a)}=0$}\\ 
\delta_{10^{a-1}s^{(6)}} \otimes \delta_{\sigma(s,a)}\otimes
\mathcal{U}(\left\{ 0,\dots,9 \right\})&\text{si $s^{(a)}=1$}.
\end{cases}$$

On considère un paiement non-escompté ($\gamma=1$, même si cela ne rentre pas strictement dans le cadre théorique du cours) et une distribution pour l'état initial $\mu=\delta_{(1,1,1,1,1)}\otimes \mathcal{U}(\left\{ 0,\dots,9 \right\})$.

In [7]:
import random
import math
import numpy as np


# Liste contenant les états sous la forme de tuples
S = [(i1,i2,i3,i4,i5,i6) for i1 in [0,1] for i2 in [0,1] for i3 in [0,1] for i4 in [0,1] for i5 in [0,1] for i6 in range(10)]

**Question 1 :** Compléter les fonctions suivantes qui calculent $B_\pi$ et $B_*$ dans l'espace des fonctions état-valeur. Les arguments `v` et `pi` sont ici des dictonnaires dont les clés sont les états représentés sous la forme de tuples comme dans l'ensemble ci-dessus. On suppose qu'on ne travaille qu'avec politiques stationnaires et déterministes.

In [8]:
def transition(s,a):
    s_list = list(s)
    s_=s_list.copy()
    state=s_[:5]
    digit_to_change=s_list[a-1] # index of digit to change
    input_number=s[5] # digit to input

    if digit_to_change==0: # digit is already full
        return 0,tuple(state+[random.randint(0,9)])
    
    s_[a-1]=0
    state=s_[:5]
    return input_number*10**(a-1),tuple(state+[random.randint(0,9)]) # Piecewise rewarde because gamma=1

In [9]:
def B_pi(v,pi):
    v_ = v.copy() # v_ = v ne ferait que copier un pointeur pointant vers le même dictionnaire
    
    for state in S:
        action_taken=pi[state]
        reward,new_state=transition(state,action_taken)
        v_[state]=reward+v[new_state]

    return v_

def B_star(v):
    v_ = v.copy()
    # compléter
        
    for state in S:
        action_value=[]
        for action in range(1,6):
            reward,new_state=transition(state,action)
            action_value.append(reward+v[new_state])
        
        v_[state]=max(action_value)
    return v_

**Question 2 :** Compléter la fonction suivante qui pour une fonction état-valeur $v$ donnée calcule la quantité
$$\mathbb{E}_{S\sim\mu}\left[ v(S) \right].$$

In [10]:
def avg_reward(v):
    total=0

    for digit in range(10):
        s=(1,1,1,1,1,digit)
        total+=v[s]
    return total/10
    

**Question 3 :** Compléter la fonction suivante qui détermine une politique gloutonne par rapport à une fonction valeur donnée.

In [11]:
def greedy_policy(v):
    pi = dict()
    for state in S:
        action_value=[]
        for action in range(1,6):
            reward,next_state=transition(state,action)
            action_value.append(reward+v[next_state])
        best_action= action_value.index(max(action_value))
        pi[state]=best_action+1
    
    return pi

**Question 4 :**  En utilisant une itération valeur (resp. une itération de politique), déterminer une politique optimale $\pi^*$, ainsi que le paiement associé
$\mathbb{E}_{\mu,\pi^*}\left[ \sum_{t=1}^{+\infty}R_t\right].$ Y inclure un critère d'arrêt en supposant que l'optimalité sera atteinte en un nombre fini d'itérations. On partira d'une fonction valeur initiale nulle.

In [12]:
## Value Iteration 
v = dict()
for s in S:
    v[s] = 0.


for iteration in range(100):
    v_old=v.copy()
    v=B_star(v)

    if max(abs(v[s] - v_old[s]) for s in S) < 1e-6:
        print("Convergence atteinte à l'itération", iteration)
        break

pi=greedy_policy(v)

total_reward=avg_reward(v)

print(f"Total reward: {total_reward}")



Convergence atteinte à l'itération 43
Total reward: 99994.5


In [13]:
## policy iteration

v = dict()
for s in S:
    v[s] = 0.

pi=greedy_policy(v)
iteration=0

while True:
    pi_old=pi.copy()

    for _ in range(10):
        v_old = v.copy()
        v = B_pi(v,pi)
        if max(abs(v[s]-v_old[s]) for s in S) < 1e-6:
            break

    pi=greedy_policy(v)

    if pi==pi_old:
        print("Convergence atteinte à l'itération", iteration)
        break
    iteration+=1

v=B_pi(v,pi)
total_reward=avg_reward(v)
print(f"Total reward: {total_reward}")


Convergence atteinte à l'itération 20
Total reward: 99994.5


In [14]:
average_reward(policy=pi_star)
# much better results

74376.5379

# Exercice 2 - Itérations asynchrones pour le plus grand nombre à 5 chiffres

On reprend le problème du plus grand nombre à 5 chiffres. On prendra toujours pour fonction valeur initiale un vecteur nul.

In [15]:
import random
import math
import numpy as np

def sigma(s,a):
    number = list(s[:5])
    number[a-1] = 0
    return number

S = [(i1,i2,i3,i4,i5,i6) for i1 in [1,0] for i2 in [1,0] for i3 in [1,0] for i4 in [1,0] for i5 in [1,0] for i6 in range(10)]

def B_star(v):
    v_ = v.copy()
    for s in S:
        values = [0]
        for a in [1,2,3,4,5]:
            if s[a-1] == 1:
                number = sigma(s,a)
                values.append(10**(a-1)*s[5] + sum([v[tuple(number+[i])] for i in range(10)])/10)
        v_[s] = max(values)
    return v_

**Question 1**: Combien de mises à jours sont nécessaires à une itération état-valeur synchrone pour atteindre une fonction valeur optimale ?

In [16]:
v = dict()
for s in S:
    v[s] = 0.
n=0

# compléter

**Question 2**: Implémenter une itération état-valeur asynchrone qui met à jour les valeurs des états de façon cyclique. Trouver un ordre de mise à jour d'états minimisant le nombre de mises à jour nécessaires pour atteindre une fonction valeur optimale.

In [17]:
v = dict()
for s in S:
    v[s] = 0.
n=0
nb_updates = 0

# compléter